# SmolVLA Post-Training on NutAssembly + Evaluation with Video

This notebook fine-tunes **SmolVLA** (450M params) on RobotSuite's **NutAssembly** task
(both round and square nuts) and then evaluates it with **MP4 video output** for every
test scenario.

## Pipeline
1. **Setup** — Install deps, configure headless rendering
2. **Data** — Download robomimic demos (500 human teleoperated demos: 300 mh + 200 ph)
3. **Convert** — Build local LeRobot dataset
4. **Train** — Fine-tune SmolVLA via `lerobot-train`
5. **Evaluate** — Test on NutAssembly, record video for every episode
6. **Review** — Watch videos, see which scenarios succeeded/failed

**Note:** Scripted demo collection is skipped — the proportional controller has 0%
success rate on the full two-nut NutAssembly task.

**Requirements:** GPU runtime (A100 recommended, T4 works with smaller batch size)

Runtime → Change runtime type → **T4 GPU** or **A100 GPU**

---
## 1. System Setup & Dependencies

In [ ]:
%%bash
# Install system dependencies for headless MuJoCo rendering
apt-get update -qq
apt-get install -y -qq libegl1-mesa-dev libgl1-mesa-glx libosmesa6-dev libglfw3 ffmpeg patchelf > /dev/null 2>&1

# Create NVIDIA EGL ICD config (Colab is missing this by default)
mkdir -p /usr/share/glvnd/egl_vendor.d
cat > /usr/share/glvnd/egl_vendor.d/10_nvidia.json << 'EOF'
{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}
EOF

echo "System dependencies installed."

In [ ]:
# Install Python packages for NutAssembly pipeline
# IMPORTANT: Do NOT reinstall torch/torchvision — Colab has CUDA-optimized versions pre-installed.

# 1) Non-ML dependencies
!pip install -q robosuite imageio[ffmpeg] matplotlib h5py Pillow pandas

# 2) Install LeRobot WITH [smolvla] extra from source
# The [smolvla] extra installs transformers>=4.57.1,<5.0.0 which is compatible
# with lerobot's huggingface-hub>=0.34.2,<0.36.0 pin.
# Without it, Colab's transformers 5.0.0 stays and conflicts with hf-hub 0.35.x.
!pip install -q "lerobot[smolvla] @ git+https://github.com/huggingface/lerobot.git"

# 3) Pin numpy to exact 2.0.2 (mujoco needs >=2.0; numba needs <2.1; tensorflow needs <2.2)
!pip install -q numpy==2.0.2

print("\nPackages installed. Restarting runtime to fix numpy C bindings...")
print("After restart, SKIP this cell and continue from the next section.")

# Restart runtime to clear stale numpy C bindings
import os
os.kill(os.getpid(), 9)

### After runtime restart — continue from here

In [ ]:
import os

# MUST be set BEFORE importing mujoco or robosuite
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["MUJOCO_EGL_DEVICE_ID"] = "0"

# Fix robosuite macros_private warning
import robosuite
macros_private = os.path.join(os.path.dirname(robosuite.__file__), "macros_private.py")
if not os.path.exists(macros_private):
    with open(macros_private, "w") as f:
        f.write("# Auto-generated private macros\n")

import torch
import numpy as np

print(f"robosuite {robosuite.__version__}")
print(f"torch {torch.__version__}")
print(f"numpy {np.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")

In [ ]:
# Clone the project repo (or mount Google Drive)
import os

REPO_DIR = "/content/AUTOLAB-Project/robosuite-vla"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/williampacini/AUTOLAB-Project.git /content/AUTOLAB-Project
    !cd /content/AUTOLAB-Project && git checkout claude/train-test-smolvla-robotics-sjmUY

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")
!ls -la

---
## 2. Quick Environment Test

Verify NutAssembly renders correctly before collecting data.

In [ ]:
import robosuite as suite
import numpy as np
import matplotlib.pyplot as plt

# Quick test: create NutAssembly env and render one frame
env = suite.make(
    env_name="NutAssembly",
    robots="Panda",
    has_renderer=False,
    has_offscreen_renderer=True,
    use_camera_obs=True,
    use_object_obs=True,
    camera_names=["agentview", "robot0_eye_in_hand"],
    camera_heights=256,
    camera_widths=256,
    reward_shaping=True,
    single_object_mode=0,  # Both nuts
    horizon=1000,
)

obs = env.reset()

# Show both camera views
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))
ax1.imshow(np.flip(obs["agentview_image"], axis=0))
ax1.set_title("Agentview")
ax1.axis("off")

ax2.imshow(np.flip(obs["robot0_eye_in_hand_image"], axis=0))
ax2.set_title("Wrist Camera")
ax2.axis("off")

plt.suptitle("NutAssembly Environment — Both Nuts", fontsize=14)
plt.tight_layout()
plt.show()

# Print observation keys
print("\nObservation keys:")
for key in sorted(obs.keys()):
    if isinstance(obs[key], np.ndarray):
        print(f"  {key:35s} shape={obs[key].shape}")

env.close()
print("\nEnvironment test passed!")

---
## 3. Data Collection — Scripted Policy (SKIPPED)

The scripted two-nut pick-and-place policy has a 0% success rate on the full
NutAssembly task (both round + square nuts). The proportional controller cannot
reliably complete the dual-nut assembly within the episode horizon.

**We skip this step** and rely entirely on the 500 robomimic human demonstrations
(300 mh + 200 ph) which are high-quality teleoperated data.

In [ ]:
# Scripted collection SKIPPED — 0% success rate on two-nut NutAssembly.
# The proportional controller cannot reliably complete both nut placements.
# If you want to try anyway (may take hours with no results):
# !python data/collect_nut_assembly.py --num-demos 50 --output data/nut_assembly_scripted/demos.hdf5
print("Scripted collection skipped — using robomimic human demos only.")

In [ ]:
# Display preview GIF of scripted demo
from IPython.display import Image, display
preview_path = "data/nut_assembly_scripted/preview.gif"
if os.path.exists(preview_path):
    display(Image(filename=preview_path))
    print("Preview of scripted demo")
else:
    print("No preview generated")

---
## 4. Data Download — HuggingFace robomimic (3/4 of data)

Download NutAssemblySquare + NutAssemblyRound demos from the robomimic dataset.

In [ ]:
# Download robomimic NutAssembly variants from HuggingFace
!python data/download_nut_assembly.py \
    --output-dir data/robomimic_nut_assembly \
    --max-demos 75

---
## 5. Convert & Combine into LeRobot Dataset

Convert all HDF5 sources into a single local LeRobot-format dataset.

In [ ]:
# Build the dataset from robomimic human demos only
# --skip-collect: scripted policy fails on two-nut task (0% success)
# --skip-render: v1.5 raw demos have state dim mismatch (45 vs 44)
# Uses image.hdf5 files only (skips image_abs.hdf5 duplicates automatically)
!python data/build_nut_assembly_dataset.py \
    --output-dir outputs/lerobot/nut_assembly \
    --skip-download \
    --skip-collect \
    --skip-render \
    --image-size 256

In [ ]:
# Alternatively, run the full pipeline in one go (downloads + collects + converts)
# Uncomment if you skipped steps 3-4:
# !python data/build_nut_assembly_dataset.py --output-dir outputs/lerobot/nut_assembly

In [ ]:
# Verify the dataset
import json

meta_path = "outputs/lerobot/nut_assembly/meta/info.json"
if os.path.exists(meta_path):
    with open(meta_path) as f:
        meta = json.load(f)
    print("Dataset metadata:")
    for k, v in meta.items():
        print(f"  {k}: {v}")
else:
    print("Dataset not built yet!")

---
## 6. Train SmolVLA

Fine-tune SmolVLA on the NutAssembly dataset using LeRobot's training CLI.

**Estimated time:** ~3-4 hours on A100, ~6-8 hours on T4

In [ ]:
import torch

# Auto-detect GPU and adjust batch size
if torch.cuda.is_available():
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if gpu_mem_gb > 30:  # A100
        BATCH_SIZE = 4
        print(f"A100 detected ({gpu_mem_gb:.0f} GB) — batch_size={BATCH_SIZE}")
    else:  # T4 or smaller
        BATCH_SIZE = 2
        print(f"GPU ({gpu_mem_gb:.0f} GB) — batch_size={BATCH_SIZE}")
else:
    BATCH_SIZE = 1
    print("No GPU — batch_size=1 (will be very slow!)")

DATASET_DIR = "outputs/lerobot/nut_assembly"
OUTPUT_DIR = "outputs/checkpoints/smolvla_nut_assembly"
TOTAL_STEPS = 50000

In [ ]:
# Launch training
!lerobot-train \
    --policy.type=smolvla \
    --policy.load_vlm_weights=true \
    --dataset.local_files_only=true \
    --dataset.root={DATASET_DIR} \
    --batch_size={BATCH_SIZE} \
    --steps={TOTAL_STEPS} \
    --output_dir={OUTPUT_DIR} \
    --save_freq=5000 \
    --eval_freq=2500 \
    --log_freq=100 \
    --seed=42 \
    --policy.device=cuda \
    --policy.dtype=bf16

In [ ]:
# Post-training fix: set n_action_steps=50 in config.json
# Without this, inference is 50x slower
!python train/train_smolvla.py --fix-config {OUTPUT_DIR}

---
## 7. Evaluate — Test on NutAssembly with Video Output

Every test episode is recorded as an MP4 video (3x speed) and tagged with success/fail.
This lets you visually inspect exactly which scenarios the model completed vs failed.

In [ ]:
# Run evaluation — records MP4 video for EVERY episode
!python eval/eval_nut_assembly.py \
    --checkpoint {OUTPUT_DIR} \
    --episodes 20 \
    --max-steps 1000 \
    --speedup 3 \
    --output-dir outputs \
    --camera-size 256

---
## 8. Review Results — Watch Videos & Check Success/Failure

In [ ]:
# Load evaluation results
import json
import pandas as pd

results_path = "outputs/results/nut_assembly_eval.json"
with open(results_path) as f:
    results = json.load(f)

# Summary
print(f"Task: {results['task']}")
print(f"Success Rate: {results['n_success']}/{results['n_episodes']} ({results['success_rate']:.1%})")
print(f"Video speedup: {results['video_speedup']}x")
print()

# Per-episode results table
df = pd.DataFrame(results['episodes'])
df['status'] = df['success'].map({True: 'SUCCESS', False: 'FAIL'})
print(df[['scenario_idx', 'seed', 'status', 'total_reward', 'steps']].to_string(index=False))

In [ ]:
# Display videos inline — all episodes
from IPython.display import Video, display, HTML
from pathlib import Path
import base64

video_dir = Path("outputs/videos/nut_assembly")
videos = sorted(video_dir.glob("*.mp4"))

print(f"Found {len(videos)} evaluation videos\n")

for video_path in videos:
    name = video_path.stem
    is_success = "success" in name
    color = "green" if is_success else "red"
    status = "SUCCESS" if is_success else "FAIL"

    display(HTML(f'<h3 style="color: {color}">{name} — {status}</h3>'))

    # Inline video display
    with open(video_path, "rb") as f:
        data = base64.b64encode(f.read()).decode()
    display(HTML(
        f'<video controls width="400">'
        f'<source src="data:video/mp4;base64,{data}" type="video/mp4">'
        f'</video>'
    ))
    print()

In [ ]:
# Visual summary: success/failure grid
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

episodes = results['episodes']
n = len(episodes)
cols = min(10, n)
rows = (n + cols - 1) // cols

fig, ax = plt.subplots(figsize=(max(8, cols * 1.2), max(2, rows * 1.2)))

for i, ep in enumerate(episodes):
    row = i // cols
    col = i % cols
    color = '#4CAF50' if ep['success'] else '#f44336'
    rect = mpatches.FancyBboxPatch(
        (col, rows - 1 - row), 0.9, 0.9,
        boxstyle="round,pad=0.05", facecolor=color, edgecolor='white', linewidth=2
    )
    ax.add_patch(rect)
    ax.text(col + 0.45, rows - 1 - row + 0.45, str(i),
            ha='center', va='center', fontsize=10, color='white', fontweight='bold')

ax.set_xlim(-0.1, cols + 0.1)
ax.set_ylim(-0.1, rows + 0.1)
ax.set_aspect('equal')
ax.axis('off')

success_patch = mpatches.Patch(color='#4CAF50', label=f"Success ({results['n_success']})")
fail_patch = mpatches.Patch(color='#f44336', label=f"Fail ({results['n_fail']})")
ax.legend(handles=[success_patch, fail_patch], loc='upper right', fontsize=12)

plt.title(f"NutAssembly Evaluation: {results['success_rate']:.1%} Success Rate", fontsize=14)
plt.tight_layout()
plt.savefig("outputs/results/nut_assembly_grid.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Quick-access: show only FAILED scenarios for debugging
failed = [ep for ep in results['episodes'] if not ep['success']]
print(f"\nFailed scenarios: {len(failed)}/{len(results['episodes'])}")
print()

for ep in failed:
    video_path = Path(ep['video_path'])
    print(f"  Scenario {ep['scenario_idx']} (seed={ep['seed']}): "
          f"reward={ep['total_reward']:.2f}, steps={ep['steps']}")
    print(f"    Video: {video_path.name}")

    # Display failed video inline
    if video_path.exists():
        display(HTML(f'<h4 style="color: red">FAIL — Scenario {ep["scenario_idx"]}</h4>'))
        with open(video_path, "rb") as f:
            data = base64.b64encode(f.read()).decode()
        display(HTML(
            f'<video controls width="400">'
            f'<source src="data:video/mp4;base64,{data}" type="video/mp4">'
            f'</video>'
        ))
        print()

---
## 9. (Optional) Save to Google Drive

In [ ]:
# Uncomment to save checkpoints and videos to Drive
# from google.colab import drive
# drive.mount('/content/drive')
#
# import shutil
# drive_dir = "/content/drive/MyDrive/smolvla_nut_assembly"
# os.makedirs(drive_dir, exist_ok=True)
#
# # Copy results, videos, and checkpoint
# shutil.copytree("outputs/results", f"{drive_dir}/results", dirs_exist_ok=True)
# shutil.copytree("outputs/videos/nut_assembly", f"{drive_dir}/videos", dirs_exist_ok=True)
# shutil.copytree(OUTPUT_DIR, f"{drive_dir}/checkpoint", dirs_exist_ok=True)
# print(f"Saved to {drive_dir}")